In [0]:
# ===========================================
# Notebook Name:
# 02_identify_result_fetch_targets
#
# Purpose:
# Identify tournaments whose result API
# should be fetched or re-fetched.
#
# Inputs:
# pokemon.silver.tournaments
# pokemon.bronze.event_result_raw
# pokemon.bronze.scrape_log
# pokemon.silver.tournament_results
#
# Output:
# Temporary view: result_fetch_targets
#
# Fetch reasons:
# - never_fetched
# - retry_error
# - retry_empty
# - tournament_updated
# ===========================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TOURNAMENTS_TABLE = (
    "pokemon.silver.tournaments"
)

EVENT_RESULT_RAW_TABLE = (
    "pokemon.bronze.event_result_raw"
)

SCRAPE_LOG_TABLE = (
    "pokemon.bronze.scrape_log"
)

TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)

TARGET_VIEW = "result_fetch_targets"

MIN_EVENT_DATE = "2026-01-01"

# 終了前の大会は結果取得対象にしない
ONLY_FINISHED_EVENTS = True

# retry_emptyを毎日再取得すると無駄が出るため、
# 前回取得から最低何時間空けるか
EMPTY_RETRY_INTERVAL_HOURS = 24

# 一度に取得する大会数の安全上限
MAX_FETCH_TARGETS = 500

print("Target view:", TARGET_VIEW)

In [0]:
required_tables = [
    TOURNAMENTS_TABLE,
    EVENT_RESULT_RAW_TABLE,
]

for table_name in required_tables:
    if not spark.catalog.tableExists(
        table_name
    ):
        raise ValueError(
            f"Required table does not exist: "
            f"{table_name}"
        )

print(
    "Validation passed: "
    "required tables exist"
)

In [0]:
tournament_columns = {
    field.name
    for field in (
        spark.table(
            TOURNAMENTS_TABLE
        ).schema.fields
    )
}

print(
    "Tournament columns:",
    sorted(tournament_columns),
)

if "event_title" in tournament_columns:
    event_title_column = "event_title"

elif "tournament_name" in tournament_columns:
    event_title_column = "tournament_name"

else:
    event_title_column = None


if "event_date" not in tournament_columns:
    raise ValueError(
        f"{TOURNAMENTS_TABLE} "
        "does not contain event_date"
    )

if "last_changed_at" in tournament_columns:
    tournament_changed_column = (
        "last_changed_at"
    )

elif "updated_at" in tournament_columns:
    tournament_changed_column = (
        "updated_at"
    )

elif "source_fetched_at" in tournament_columns:
    tournament_changed_column = (
        "source_fetched_at"
    )

else:
    tournament_changed_column = None

tournament_select_columns = [
    F.col("tournament_id"),
    F.col("event_date"),
]

if event_title_column:
    tournament_select_columns.append(
        F.col(
            event_title_column
        ).alias("event_title")
    )
else:
    tournament_select_columns.append(
        F.lit(None)
        .cast("string")
        .alias("event_title")
    )

if tournament_changed_column:
    tournament_select_columns.append(
        F.col(
            tournament_changed_column
        ).alias(
            "tournament_changed_at"
        )
    )
else:
    tournament_select_columns.append(
        F.lit(None)
        .cast("timestamp")
        .alias(
            "tournament_changed_at"
        )
    )

for column_name in [
    "result_count",
    "event_ended_at",
    "event_started_at",
    "event_type_id",
    "event_type_title",
]:
    if column_name in tournament_columns:
        tournament_select_columns.append(
            F.col(column_name)
        )

tournaments_df = (
    spark.table(
        TOURNAMENTS_TABLE
    )
    .select(
        *tournament_select_columns
    )
    .filter(
        F.col("event_date")
        >= F.lit(
            MIN_EVENT_DATE
        ).cast("date")
    )
)

if ONLY_FINISHED_EVENTS:
    tournaments_df = (
        tournaments_df
        .filter(
            F.col("event_date")
            <= F.current_date()
        )
    )

print(
    "Eligible tournament count:",
    tournaments_df.count(),
)

display(
    tournaments_df
    .orderBy(
        F.col("event_date").desc(),
        "tournament_id",
    )
    .limit(20)
)

In [0]:
event_raw_df = spark.table(
    EVENT_RESULT_RAW_TABLE
)

event_raw_columns = {
    field.name
    for field in (
        event_raw_df.schema.fields
    )
}

print(
    "Event result raw columns:",
    sorted(event_raw_columns),
)

if "tournament_id" in event_raw_columns:
    raw_tournament_id_column = (
        "tournament_id"
    )

elif "event_id" in event_raw_columns:
    raw_tournament_id_column = (
        "event_id"
    )

else:
    raise ValueError(
        f"{EVENT_RESULT_RAW_TABLE} "
        "does not contain tournament_id "
        "or event_id"
    )

if "scraped_at" in event_raw_columns:
    raw_fetched_at_column = "scraped_at"

elif "fetched_at" in event_raw_columns:
    raw_fetched_at_column = "fetched_at"

elif "source_scraped_at" in event_raw_columns:
    raw_fetched_at_column = (
        "source_scraped_at"
    )

else:
    raise ValueError(
        f"{EVENT_RESULT_RAW_TABLE} "
        "does not contain a fetch timestamp"
    )
latest_raw_window = (
    Window
    .partitionBy("tournament_id")
    .orderBy(
        F.col(
            "result_fetched_at"
        ).desc()
    )
)

raw_select_columns = [
    F.col(
        raw_tournament_id_column
    )
    .cast("int")
    .alias("tournament_id"),

    F.col(
        raw_fetched_at_column
    )
    .alias("result_fetched_at"),
]

if "response_hash" in event_raw_columns:
    raw_select_columns.append(
        F.col("response_hash")
        .alias(
            "latest_result_response_hash"
        )
    )
else:
    raw_select_columns.append(
        F.lit(None)
        .cast("string")
        .alias(
            "latest_result_response_hash"
        )
    )

if "payload" in event_raw_columns:
    raw_select_columns.append(
        F.col("payload")
        .alias("latest_result_payload")
    )
else:
    raw_select_columns.append(
        F.lit(None)
        .cast("string")
        .alias("latest_result_payload")
    )

latest_result_raw_df = (
    event_raw_df
    .select(
        *raw_select_columns
    )
    .filter(
        F.col(
            "tournament_id"
        ).isNotNull()
    )
    .withColumn(
        "raw_row_number",
        F.row_number().over(
            latest_raw_window
        ),
    )
    .filter(
        F.col("raw_row_number") == 1
    )
    .drop("raw_row_number")
)

In [0]:
if spark.catalog.tableExists(
    TOURNAMENT_RESULTS_TABLE
):
    result_columns = {
        field.name
        for field in (
            spark.table(
                TOURNAMENT_RESULTS_TABLE
            ).schema.fields
        )
    }

    if "tournament_id" in result_columns:
        completed_results_df = (
            spark.table(
                TOURNAMENT_RESULTS_TABLE
            )
            .select(
                F.col(
                    "tournament_id"
                ).cast("int")
            )
            .filter(
                F.col(
                    "tournament_id"
                ).isNotNull()
            )
            .distinct()
            .withColumn(
                "has_silver_results",
                F.lit(True),
            )
        )

    else:
        raise ValueError(
            f"{TOURNAMENT_RESULTS_TABLE} "
            "does not contain tournament_id"
        )

else:
    completed_results_df = (
        spark.createDataFrame(
            [],
            """
            tournament_id INT,
            has_silver_results BOOLEAN
            """,
        )
    )

print(
    "Tournaments with Silver results:",
    completed_results_df.count(),
)

In [0]:
if spark.catalog.tableExists(
    SCRAPE_LOG_TABLE
):
    scrape_log_df = spark.table(
        SCRAPE_LOG_TABLE
    )

    scrape_log_columns = {
        field.name
        for field in (
            scrape_log_df.schema.fields
        )
    }

    print(
        "Scrape log columns:",
        sorted(scrape_log_columns),
    )

else:
    scrape_log_df = None
    scrape_log_columns = set()

latest_error_log_df = (
    spark.createDataFrame(
        [],
        """
        tournament_id INT,
        latest_fetch_status STRING,
        latest_fetch_error STRING,
        latest_log_at TIMESTAMP
        """,
    )
)

if scrape_log_df is not None:
    if "tournament_id" in scrape_log_columns:
        log_id_expr = (
            F.col("tournament_id")
            .cast("int")
        )

    elif "source_id" in scrape_log_columns:
        log_id_expr = (
            F.col("source_id")
            .cast("int")
        )

    else:
        log_id_expr = None

    if "scraped_at" in scrape_log_columns:
        log_time_column = "scraped_at"

    elif "created_at" in scrape_log_columns:
        log_time_column = "created_at"

    elif "started_at" in scrape_log_columns:
        log_time_column = "started_at"

    else:
        log_time_column = None

    if (
        log_id_expr is not None
        and log_time_column is not None
        and "status" in scrape_log_columns
    ):
        filtered_log_df = scrape_log_df

        if "source_type" in scrape_log_columns:
            filtered_log_df = (
                filtered_log_df
                .filter(
                    F.col("source_type").isin(
                        "event_result",
                        "tournament_result",
                        "event_results",
                    )
                )
            )

        log_select_columns = [
            log_id_expr.alias(
                "tournament_id"
            ),
            F.col("status").alias(
                "latest_fetch_status"
            ),
            F.col(
                log_time_column
            ).alias("latest_log_at"),
        ]

        if "error" in scrape_log_columns:
            log_select_columns.append(
                F.col("error").alias(
                    "latest_fetch_error"
                )
            )

        elif "error_message" in scrape_log_columns:
            log_select_columns.append(
                F.col(
                    "error_message"
                ).alias(
                    "latest_fetch_error"
                )
            )

        else:
            log_select_columns.append(
                F.lit(None)
                .cast("string")
                .alias(
                    "latest_fetch_error"
                )
            )
        latest_log_window = (
            Window
            .partitionBy(
                "tournament_id"
            )
            .orderBy(
                F.col(
                    "latest_log_at"
                ).desc()
            )
        )

        latest_error_log_df = (
            filtered_log_df
            .select(
                *log_select_columns
            )
            .filter(
                F.col(
                    "tournament_id"
                ).isNotNull()
            )
            .withColumn(
                "log_row_number",
                F.row_number().over(
                    latest_log_window
                ),
            )
            .filter(
                F.col(
                    "log_row_number"
                ) == 1
            )
            .drop(
                "log_row_number"
            )
        )

In [0]:
result_status_df = (
    tournaments_df.alias("t")
    .join(
        latest_result_raw_df.alias("r"),
        on="tournament_id",
        how="left",
    )
    .join(
        completed_results_df.alias("s"),
        on="tournament_id",
        how="left",
    )
    .join(
        latest_error_log_df.alias("l"),
        on="tournament_id",
        how="left",
    )
    .withColumn(
        "has_raw_result",
        F.col(
            "result_fetched_at"
        ).isNotNull(),
    )
    .withColumn(
        "has_silver_results",
        F.coalesce(
            F.col(
                "has_silver_results"
            ),
            F.lit(False),
        ),
    )
)

In [0]:
result_status_df = (
    result_status_df
    .withColumn(
        "empty_retry_available_at",
        F.expr(
            f"""
            result_fetched_at
            + INTERVAL
            {EMPTY_RETRY_INTERVAL_HOURS}
            HOURS
            """
        ),
    )
    .withColumn(
        "can_retry_empty",
        F.col(
            "result_fetched_at"
        ).isNotNull()
        & (
            F.current_timestamp()
            >= F.col(
                "empty_retry_available_at"
            )
        ),
    )
)

In [0]:
error_status_values = [
    "error",
    "failed",
    "failure",
    "http_error",
]

result_targets_base_df = (
    result_status_df
    .withColumn(
        "fetch_reason",
        F.when(
            F.lower(
                F.coalesce(
                    F.col(
                        "latest_fetch_status"
                    ),
                    F.lit(""),
                )
            ).isin(
                *error_status_values
            ),
            F.lit("retry_error"),
        )
        .when(
            ~F.col("has_raw_result"),
            F.lit("never_fetched"),
        )
        .when(
            F.col(
                "tournament_changed_at"
            ).isNotNull()
            & F.col(
                "result_fetched_at"
            ).isNotNull()
            & (
                F.col(
                    "tournament_changed_at"
                )
                > F.col(
                    "result_fetched_at"
                )
            ),
            F.lit(
                "tournament_updated"
            ),
        )
        .when(
            ~F.col(
                "has_silver_results"
            )
            & F.col(
                "can_retry_empty"
            ),
            F.lit("retry_empty"),
        ),
    )
)

result_fetch_targets_df = (
    result_targets_base_df
    .filter(
        F.col(
            "fetch_reason"
        ).isNotNull()
    )
)

In [0]:
result_fetch_targets_df = (
    result_fetch_targets_df
    .withColumn(
        "priority",
        F.when(
            F.col("fetch_reason")
            == "retry_error",
            F.lit(1),
        )
        .when(
            F.col("fetch_reason")
            == "never_fetched",
            F.lit(2),
        )
        .when(
            F.col("fetch_reason")
            == "tournament_updated",
            F.lit(3),
        )
        .when(
            F.col("fetch_reason")
            == "retry_empty",
            F.lit(4),
        )
        .otherwise(
            F.lit(99)
        ),
    )
)

In [0]:
if "result_count" in tournament_columns:
    result_fetch_targets_df = (
        result_fetch_targets_df
        .withColumn(
            "has_reported_results",
            F.coalesce(
                F.col(
                    "result_count"
                ),
                F.lit(0),
            ) > 0,
        )
    )

else:
    result_fetch_targets_df = (
        result_fetch_targets_df
        .withColumn(
            "has_reported_results",
            F.lit(None)
            .cast("boolean"),
        )
    )

In [0]:
result_fetch_targets_df = (
    result_fetch_targets_df
    .select(
        "tournament_id",
        "event_date",
        "event_title",
        "fetch_reason",
        "priority",
        "has_reported_results",
        "has_raw_result",
        "has_silver_results",
        "result_fetched_at",
        "tournament_changed_at",
        "latest_fetch_status",
        "latest_fetch_error",
        "latest_result_response_hash",
        "empty_retry_available_at",
    )
    .orderBy(
        F.col("priority").asc(),
        F.col(
            "has_reported_results"
        ).desc_nulls_last(),
        F.col("event_date").desc(),
        F.col("tournament_id").desc(),
    )
    .limit(
        MAX_FETCH_TARGETS
    )
)

In [0]:
result_fetch_targets_df.createOrReplaceTempView(
    TARGET_VIEW
)

print(
    f"Temporary view created: "
    f"{TARGET_VIEW}"
)

In [0]:
target_count = (
    result_fetch_targets_df.count()
)

print(
    "Result fetch target count:",
    target_count,
)

display(
    result_fetch_targets_df
)

In [0]:
display(
    result_fetch_targets_df
    .groupBy(
        "fetch_reason"
    )
    .agg(
        F.count("*").alias(
            "target_count"
        ),
        F.min(
            "event_date"
        ).alias(
            "oldest_event_date"
        ),
        F.max(
            "event_date"
        ).alias(
            "latest_event_date"
        ),
    )
    .orderBy(
        "fetch_reason"
    )
)

In [0]:
display(
    result_targets_base_df
    .withColumn(
        "fetch_status_summary",
        F.when(
            F.col(
                "fetch_reason"
            ).isNotNull(),
            F.col(
                "fetch_reason"
            ),
        )
        .when(
            F.col(
                "has_silver_results"
            ),
            F.lit(
                "completed"
            ),
        )
        .when(
            ~F.col(
                "can_retry_empty"
            )
            & F.col(
                "has_raw_result"
            ),
            F.lit(
                "waiting_empty_retry"
            ),
        )
        .otherwise(
            F.lit(
                "not_targeted"
            )
        ),
    )
    .groupBy(
        "fetch_status_summary"
    )
    .count()
    .orderBy(
        F.col("count").desc()
    )
)

In [0]:
duplicate_targets_df = (
    result_fetch_targets_df
    .groupBy(
        "tournament_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_count = (
    duplicate_targets_df.count()
)

if duplicate_count > 0:
    display(
        duplicate_targets_df
    )

    raise ValueError(
        f"{duplicate_count} duplicate "
        "result fetch targets detected"
    )

print(
    "Validation passed: "
    "one target per tournament_id"
)

valid_fetch_reasons = [
    "never_fetched",
    "retry_error",
    "retry_empty",
    "tournament_updated",
]

invalid_reason_df = (
    result_fetch_targets_df
    .filter(
        ~F.col(
            "fetch_reason"
        ).isin(
            *valid_fetch_reasons
        )
    )
)

if invalid_reason_df.count() > 0:
    display(
        invalid_reason_df
    )

    raise ValueError(
        "Invalid fetch reasons detected"
    )

print(
    "Validation passed: "
    "all fetch reasons are valid"
)

In [0]:
result_fetch_requests_df = (
    result_fetch_targets_df
    .select(
        "tournament_id",
        "fetch_reason",
        "priority",
    )
)
display(
    result_fetch_requests_df
)

result_fetch_requests_df.createOrReplaceTempView(
    "result_fetch_requests"
)

In [0]:
RESULT_FETCH_TARGETS_TABLE = (
    "pokemon.ops.result_fetch_targets"
)

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "pokemon.ops"
)

(
    result_fetch_targets_df
    .withColumn(
        "identified_at",
        F.current_timestamp(),
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        RESULT_FETCH_TARGETS_TABLE
    )
)

print(
    "Result fetch targets saved:",
    RESULT_FETCH_TARGETS_TABLE,
)

In [0]:
display(
    spark.table(
        RESULT_FETCH_TARGETS_TABLE
    )
    .orderBy(
        "priority",
        F.col(
            "event_date"
        ).desc(),
    )
)